# FireWatch TN — Entraînement du Module A sur Kaggle (GPU)

1. *Add Data* → rechercher **next-day-wildfire-spread** (fantineh)
2. *Settings* → Accelerator **GPU T4 x2** ou **P100**, Internet **On**
3. Exécuter les cellules dans l'ordre. Les résultats sont dans `/kaggle/working/runs`.

In [ ]:
!git clone -q https://github.com/<votre-compte>/firewatch-tn.git
%cd firewatch-tn
!pip install -q pyyaml

In [ ]:
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))
!ls /kaggle/input/next-day-wildfire-spread

## Baseline de persistance (référence minimale à battre)

In [ ]:
DATA = '/kaggle/input/next-day-wildfire-spread'
!python -m ml.spread_tf.evaluate --baseline persistence --set data.data_dir=$DATA train.output_dir=/kaggle/working/runs

## Entraînement du U-Net résiduel

In [ ]:
!python -m ml.spread_tf.train --set data.data_dir=$DATA train.output_dir=/kaggle/working/runs train.mixed_precision=true

## Évaluation (seuil choisi sur la validation) et export

In [ ]:
import glob
RUN = sorted(glob.glob('/kaggle/working/runs/res_unet-*'))[-1]
!python -m ml.spread_tf.evaluate --run $RUN --set data.data_dir=$DATA
!python -m ml.spread_tf.export --run $RUN --out /kaggle/working/models/spread

## Visualiser quelques prédictions

In [ ]:
import keras, numpy as np, matplotlib.pyplot as plt
from ml.spread_tf import losses, metrics, model  # objets Keras personnalisés
from ml.spread_tf.dataset import DataConfig, make_dataset
from ml.spread_tf.constants import MODEL_CHANNELS

net = keras.models.load_model(f'{RUN}/best.keras')
x, y = next(iter(make_dataset(DataConfig(data_dir=DATA, batch_size=16), 'test')))
p = tf.sigmoid(net(x)).numpy()
idx = np.argsort(-(y.numpy() == 1).sum(axis=(1, 2, 3)))[:4]
fig, ax = plt.subplots(4, 3, figsize=(9, 12))
for r, i in enumerate(idx):
    ax[r, 0].imshow(x[i, ..., MODEL_CHANNELS.index('PrevFireMask')], vmin=-1, vmax=1); ax[r, 0].set_title('Feu J')
    ax[r, 1].imshow(y[i, ..., 0], vmin=-1, vmax=1); ax[r, 1].set_title('Feu J+1 (réel)')
    ax[r, 2].imshow(p[i, ..., 0], vmin=0, vmax=1, cmap='inferno'); ax[r, 2].set_title('Probabilité prédite')
for a in ax.ravel(): a.axis('off')
plt.tight_layout()